# Auditoria de Parciais por Time

Informe a rodada e os IDs dos times para auditar. O notebook lista atletas por posição, reservas e reserva de luxo, com pontuações parciais.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import time
import requests
import pandas as pd

import scripts.parciais as parciais
import importlib
importlib.reload(parciais)

# ======= CONFIG =======
RODADA = 2
TIME_IDS = [ 48498051 ]

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*",
}
parciais.HEADERS = HEADERS

print("ROOT:", ROOT)
print("RODADA:", RODADA)
print("TIME_IDS:", TIME_IDS)

ROOT: c:\Users\ferna\Projetos\GitHub\cartola_2026
RODADA: 2
TIME_IDS: [48498051]


In [2]:
# Carrega parciais e clubes que ja jogaram
mapa_pontuados = parciais.fetch_pontuados()
clubes_jogaram = parciais.clubes_que_ja_jogaram(RODADA)
clubes_na_rodada = parciais.clubes_da_rodada(RODADA)
print("Atletas pontuados:", len(mapa_pontuados))
print("Clubes que ja jogaram:", len(clubes_jogaram))
print("Clubes na rodada:", len(clubes_na_rodada))

Atletas pontuados: 212
Clubes que ja jogaram: 12
Clubes na rodada: 18


In [3]:
def _id_int(val):
    try:
        return int(val)
    except Exception:
        return None


def atleta_row(a, origem, capitao_id=None, luxo_id=None):
    aid = _id_int(a.get("atleta_id")) if isinstance(a, dict) else None
    pos = a.get("posicao_id") if isinstance(a, dict) else None
    clube_id = _id_int(a.get("clube_id")) if isinstance(a, dict) else None
    pont = mapa_pontuados.get(aid) if aid is not None else None
    clube_na_rodada = (clube_id in clubes_na_rodada) if clube_id is not None else None
    return {
        "atleta_id": aid,
        "nome": a.get("apelido") or a.get("nome") or a.get("nome_popular") or a.get("slug") or "",
        "posicao_id": pos,
        "posicao": parciais.setor_por_posicao(pos) if pos is not None else "",
        "clube_id": clube_id,
        "pontuacao": pont,
        "origem": origem,
        "capitao": (aid is not None and capitao_id is not None and aid == capitao_id),
        "reserva_luxo": (aid is not None and luxo_id is not None and aid == luxo_id),
        "clube_jogou": (clube_id in clubes_jogaram) if clube_id is not None else None,
        "clube_na_rodada": clube_na_rodada,
    }


def nome_time_from_payload(data):
    if not isinstance(data, dict):
        return None
    t = data.get("time") if isinstance(data.get("time"), dict) else None
    if t:
        return t.get("nome") or t.get("nome_cartola") or t.get("slug")
    return data.get("time_nome")


def build_mapping(atletas_list, reservas_list):
    m = {}
    for a in atletas_list or []:
        if isinstance(a, dict):
            aid = _id_int(a.get("atleta_id"))
            if aid is not None:
                m[aid] = a
    for a in reservas_list or []:
        if isinstance(a, dict):
            aid = _id_int(a.get("atleta_id"))
            if aid is not None:
                m.setdefault(aid, a)
    return m

In [4]:
for tid in TIME_IDS:
    print("\n=============================")
    print("TIME ID:", tid)

    data = parciais.fetch_time_payload(int(tid), RODADA)
    time_nome = nome_time_from_payload(data) or f"Time {tid}"
    print("NOME:", time_nome)

    atletas = data.get("atletas") if isinstance(data, dict) else []
    reservas = data.get("reservas") if isinstance(data, dict) else []

    capitao_id = _id_int(data.get("capitao_id"))
    if capitao_id is None and isinstance(data.get("time"), dict):
        capitao_id = _id_int(data["time"].get("capitao_id"))

    luxo_id = _id_int(data.get("reserva_luxo_id"))

    titulares = []
    tecnico = None
    for a in atletas or []:
        if not isinstance(a, dict):
            continue
        pos = a.get("posicao_id")
        if pos == 6:
            tecnico = a
        else:
            titulares.append(a)

    rows_titulares = [atleta_row(a, "titular", capitao_id, luxo_id) for a in titulares]
    if isinstance(tecnico, dict):
        rows_titulares.append(atleta_row(tecnico, "tecnico", capitao_id, luxo_id))
    rows_reservas = [atleta_row(a, "reserva", capitao_id, luxo_id) for a in (reservas or [])]

    df_titulares = pd.DataFrame(rows_titulares).sort_values(["posicao_id", "nome"], na_position="last")
    df_reservas = pd.DataFrame(rows_reservas).sort_values(["posicao_id", "nome"], na_position="last")

    print("\nTitulares / Tecnico")
    display(df_titulares)

    print("\nReservas")
    display(df_reservas)

    # Calcula parcial pela regra padrao
    total, subs_banco, sub_luxo = parciais.calcular_parcial_time_detalhado(
        int(tid), RODADA, mapa_pontuados, clubes_jogaram
    )

    print("\nTOTAL PARCIAL (calculado):", total)
    print("SUBS BANCO:", subs_banco)
    print("SUB LUXO:", sub_luxo)

    # Resolve nomes das substituicoes
    mapping = build_mapping(atletas, reservas)
    def atleta_nome(aid):
        a = mapping.get(_id_int(aid))
        if not isinstance(a, dict):
            return str(aid)
        return a.get("apelido") or a.get("nome") or str(aid)

    if subs_banco:
        print("\nDetalhe subs banco:")
        for sub in subs_banco:
            if len(sub) >= 2:
                out_id, in_id = sub[0], sub[1]
                print(f"- saiu {atleta_nome(out_id)} ({out_id}) -> entrou {atleta_nome(in_id)} ({in_id})")

    if sub_luxo:
        out_id, in_id, setor = sub_luxo
        print(f"\nLuxo: saiu {atleta_nome(out_id)} ({out_id}) -> entrou {atleta_nome(in_id)} ({in_id}) | setor={setor}")

    time.sleep(0.3)


TIME ID: 48498051
NOME: Pity10

Titulares / Tecnico


,atleta_id,nome,posicao_id,posicao,clube_id,pontuacao,origem,capitao,reserva_luxo,clube_jogou,clube_na_rodada
3,101574,Hugo Souza,1,Goleiro,264,NaN,titular,False,False,False,False
9,105175,Khellven,2,Laterais,275,2.60,titular,False,False,True,True
1,102130,Piquerez,2,Laterais,275,12.80,titular,False,False,True,True
4,71684,Gustavo Gómez,3,Zagueiros,275,8.50,titular,False,False,True,True
10,97653,Murilo,3,Zagueiros,275,16.00,titular,False,False,True,True
7,122394,Gustavinho,4,Meias,280,12.20,titular,False,False,True,True
2,105647,Maurício,4,Meias,275,11.70,titular,False,False,True,True
5,142279,Nuno Moreira,4,Meias,267,NaN,titular,False,False,False,True
6,116706,Andrés Gómez,5,Atacantes,267,NaN,titular,False,False,False,True
0,109401,Erick Pulga,5,Atacantes,265,NaN,titular,False,False,False,True



Reservas


,atleta_id,nome,posicao_id,posicao,clube_id,pontuacao,origem,capitao,reserva_luxo,clube_jogou,clube_na_rodada
3,51413,Walter,1,Goleiro,2305,4.5,reserva,False,False,True,True
0,110715,Andrés Hurtado,2,Laterais,280,5.5,reserva,False,False,True,True
4,122441,Kayky Almeida,3,Zagueiros,364,1.5,reserva,False,False,True,True
2,94067,Matheus Fernandes,4,Meias,280,2.8,reserva,False,False,True,True
1,86757,Everton,5,Atacantes,262,3.4,reserva,False,True,True,True



TOTAL PARCIAL (calculado): 78.75
SUBS BANCO: [(101574, 51413, 1)]
SUB LUXO: None

Detalhe subs banco:
- saiu Hugo Souza (101574) -> entrou Walter (51413)
